# Phase 3: Industrial Robustness Adaptation, Mitigation & Recovery
**M.Tech Dissertation**: *Robustness and Reproducibility of Open-Source Small Multimodal LLMs for Industrial Anomaly Detection*

## 🎯 Overview & Objectives
Following Phase 2 findings where factory floor corruptions (Conveyor Motion Blur  = +1.78$, Gaussian Sensor Noise  = +0.89$) caused severe degradation,
Phase 3 implements and benchmarks three edge-deployable mitigation strategies:
1. **Input-Level Test-Time Image Restoration (TTA-IR)**: Laplacian Unsharp Masking, Bilateral Filtering, CLAHE.
2. **Prompt-Level Defect-Anchored Reasoning (DAR)**: Domain noise-invariant instructions directing model attention to micro-defects.
3. **Hybrid Compound Adaptation**: Combined pre-filtering + anchored reasoning.

Metrics computed include **Robustness Recovery Rate (\%$)**, **Degradation Slope Flattening ($\Delta RDS$)**, and **Subtask Recovery Matrix**.

### CELL 1: Environment & Module Setup

In [ ]:
import os
import sys
import json
import time
import random
from pathlib import Path
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch

# Add phase-2 and phase-3 modules
CURRENT_DIR = Path.cwd()
PARENT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "phase-3" else CURRENT_DIR / "phase-3"
sys.path.insert(0, str(PARENT_DIR))
sys.path.insert(0, str(PARENT_DIR.parent / "phase-2"))

from corruptions import apply_corruption, CORRUPTION_DISPLAY_NAMES
from mitigations import apply_mitigation_filter, build_standard_prompt, build_defect_anchored_prompt

print("✅ Successfully imported Phase 2 & Phase 3 engines!")
if torch.cuda.is_available():
    print(f"Target GPU: {torch.cuda.get_device_name(0)} ({round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)} GB)")


### CELL 2: Load MMAD Dataset & Resolve Image Paths

In [ ]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "phase-3" else Path.cwd().parent
MMAD_DIR = BASE_DIR / "MMAD"
mmad_json_path = MMAD_DIR / "mmad.json"
print(f"📂 Loading MMAD benchmark: {mmad_json_path}...")
with open(mmad_json_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data)} keys from MMAD dataset.")


### CELL 3: Load Qwen3-VL-2B-Instruct Model in FP16

In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda" if torch.cuda.is_available() else "cpu",
    trust_remote_code=True,
)
model.eval()
print("✅ Qwen3-VL-2B-Instruct loaded in FP16 successfully!")


### CELL 4: Run Robustness Adaptation Benchmark (TTA vs DAR vs Hybrid)

In [ ]:
# Execute the standardized benchmark script via CLI or directly
!python phase3_adaptation.py --sample-size 45 --corruptions motion_blur,gaussian_noise,low_light --severities 3,5


### CELL 5: Display Phase 3 Figures & Recovery Metrics

In [ ]:
from IPython.display import Image as IPyImage, display
results_dir = Path("results")
for fig_name in [
    "phase3_recovery_rate_barchart.png",
    "phase3_mitigation_before_after_comparison.png",
    "phase3_restoration_visual_samples.png",
    "phase3_subtask_recovery_heatmap.png"
]:
    p = results_dir / fig_name
    if p.exists():
        print(f"
📊 Displaying: {fig_name}")
        display(IPyImage(str(p)))
